In [0]:
import uuid
from datetime import datetime

SILVER_TABELAS = {
  "orders": {
    "sql": """
      CREATE OR REPLACE TABLE workspace.silver.orders AS
      SELECT
        order_id,
        customer_id,
        lower(trim(order_status))                       AS order_status,
        try_to_timestamp(order_purchase_timestamp)      AS order_purchase_ts,
        try_to_timestamp(order_approved_at)             AS order_approved_ts,
        try_to_timestamp(order_delivered_carrier_date)  AS order_delivered_carrier_ts,
        try_to_timestamp(order_delivered_customer_date) AS order_delivered_customer_ts,
        try_to_timestamp(order_estimated_delivery_date) AS order_estimated_delivery_ts,
        CASE
          WHEN lower(trim(order_status)) = 'delivered' AND order_delivered_customer_date IS NULL THEN TRUE
          WHEN lower(trim(order_status)) = 'canceled'  AND order_delivered_customer_date IS NOT NULL THEN TRUE
          ELSE FALSE
        END                                             AS flag_data_entrega_inconsistente,
        _arquivo_origem,
        _data_ingestao
      FROM workspace.bronze.orders
    """,
    "chave": "order_id",
    "obrigatorias": ["order_id", "customer_id", "order_status"],
    "regras": [],
    "pares_vazios": [
      ("order_purchase_timestamp",      "order_purchase_ts"),
      ("order_approved_at",             "order_approved_ts"),
      ("order_delivered_carrier_date",  "order_delivered_carrier_ts"),
      ("order_delivered_customer_date", "order_delivered_customer_ts"),
      ("order_estimated_delivery_date", "order_estimated_delivery_ts"),
    ],
    "informativos": [("pedidos_marcados", "flag_data_entrega_inconsistente")],
  },
  "customers": {
    "sql": """
      CREATE OR REPLACE TABLE workspace.silver.customers AS
      SELECT
        customer_id,
        customer_unique_id,
        customer_zip_code_prefix,
        customer_city,
        customer_state,
        _arquivo_origem,
        _data_ingestao
      FROM workspace.bronze.customers
    """,
    "chave": "customer_id",
    "obrigatorias": ["customer_id", "customer_unique_id",
                     "customer_zip_code_prefix", "customer_state"],
    "regras": [
      ("cep_com_5_digitos",   "length(customer_zip_code_prefix) <> 5"),
      ("estado_2_maiusculas", "NOT (customer_state RLIKE '^[A-Z]{2}$')"),
    ],
    "pares_vazios": [],
    "informativos": [],
  },
  "sellers": {
    "sql": """
      CREATE OR REPLACE TABLE workspace.silver.sellers AS
      SELECT
        seller_id,
        seller_zip_code_prefix,
        seller_city,
        seller_state,
        (seller_city RLIKE '[0-9/\\\\\\\\]') AS flag_cidade_suspeita,
        _arquivo_origem,
        _data_ingestao
      FROM workspace.bronze.sellers
    """,
    "chave": "seller_id",
    "obrigatorias": ["seller_id", "seller_zip_code_prefix", "seller_state"],
    "regras": [
      ("cep_com_5_digitos",   "length(seller_zip_code_prefix) <> 5"),
      ("estado_2_maiusculas", "NOT (seller_state RLIKE '^[A-Z]{2}$')"),
    ],
    "pares_vazios": [],
    "informativos": [("cidades_suspeitas", "flag_cidade_suspeita")],
  },
    "products": {
    "sql": """
      CREATE OR REPLACE TABLE workspace.silver.products AS
      SELECT
        product_id,
        product_category_name,
        try_cast(product_name_lenght        AS INT) AS product_name_length,
        try_cast(product_description_lenght AS INT) AS product_description_length,
        try_cast(product_photos_qty         AS INT) AS product_photos_qty,
        try_cast(product_weight_g           AS INT) AS product_weight_g,
        try_cast(product_length_cm          AS INT) AS product_length_cm,
        try_cast(product_height_cm          AS INT) AS product_height_cm,
        try_cast(product_width_cm           AS INT) AS product_width_cm,
        (product_category_name IS NULL)                                   AS flag_sem_categoria,
        (try_cast(product_weight_g AS INT) IS NULL
         OR try_cast(product_weight_g AS INT) = 0)                        AS flag_peso_invalido,
        _arquivo_origem,
        _data_ingestao
      FROM workspace.bronze.products
    """,
    "chave": "product_id",
    "obrigatorias": ["product_id"],
    "regras": [],
    "pares_vazios": [
      ("product_category_name",       "product_category_name"),
      ("product_name_lenght",         "product_name_length"),
      ("product_description_lenght",  "product_description_length"),
      ("product_photos_qty",          "product_photos_qty"),
      ("product_weight_g",            "product_weight_g"),
      ("product_length_cm",           "product_length_cm"),
      ("product_height_cm",           "product_height_cm"),
      ("product_width_cm",            "product_width_cm"),
    ],
    "informativos": [("sem_categoria", "flag_sem_categoria"),
                     ("peso_invalido", "flag_peso_invalido")],
  },
    "order_payments": {
    "sql": """
      CREATE OR REPLACE TABLE workspace.silver.order_payments AS
      SELECT
        order_id,
        try_cast(payment_sequential   AS INT)            AS payment_sequential,
        lower(trim(payment_type))                        AS payment_type,
        try_cast(payment_installments AS INT)            AS payment_installments,
        try_cast(payment_value        AS DECIMAL(12,2))  AS payment_value,
        (try_cast(payment_value AS DECIMAL(12,2)) = 0)   AS flag_valor_zero,
        (try_cast(payment_installments AS INT) = 0)      AS flag_parcelas_zero,
        _arquivo_origem,
        _data_ingestao
      FROM workspace.bronze.order_payments
    """,
    "chave": ["order_id", "payment_sequential"],
    "obrigatorias": ["order_id", "payment_sequential", "payment_type", "payment_value"],
    "regras": [
      ("valor_negativo", "payment_value < 0"),
      ("tipo_fora_do_dominio",
       "payment_type NOT IN ('credit_card','boleto','voucher','debit_card','not_defined')"),
    ],
    "pares_vazios": [
      ("payment_sequential",   "payment_sequential"),
      ("payment_installments", "payment_installments"),
      ("payment_value",        "payment_value"),
    ],
    "informativos": [("valor_zero", "flag_valor_zero"),
                     ("parcelas_zero", "flag_parcelas_zero")],
  },
    "order_items": {
    "sql": """
      CREATE OR REPLACE TABLE workspace.silver.order_items AS
      SELECT
        order_id,
        try_cast(order_item_id AS INT)                  AS order_item_id,
        product_id,
        seller_id,
        try_to_timestamp(shipping_limit_date)           AS shipping_limit_ts,
        try_cast(price         AS DECIMAL(12,2))        AS price,
        try_cast(freight_value AS DECIMAL(12,2))        AS freight_value,
        (try_cast(freight_value AS DECIMAL(12,2)) = 0)  AS flag_frete_zero,
        _arquivo_origem,
        _data_ingestao
      FROM workspace.bronze.order_items
    """,
    "chave": ["order_id", "order_item_id"],
    "obrigatorias": ["order_id", "order_item_id", "product_id", "seller_id",
                     "price", "freight_value"],
    "regras": [
      ("preco_nao_positivo", "price <= 0"),
      ("frete_negativo",     "freight_value < 0"),
    ],
    "pares_vazios": [
      ("order_item_id",       "order_item_id"),
      ("shipping_limit_date", "shipping_limit_ts"),
      ("price",               "price"),
      ("freight_value",       "freight_value"),
    ],
    "informativos": [("frete_zero", "flag_frete_zero")],
  },
    "order_reviews": {
    "sql": """
      CREATE OR REPLACE TABLE workspace.silver.order_reviews AS
      SELECT
        review_id,
        order_id,
        try_cast(review_score AS INT)                  AS review_score,
        review_comment_title,
        review_comment_message,
        try_to_timestamp(review_creation_date)         AS review_creation_ts,
        try_to_timestamp(review_answer_timestamp)      AS review_answer_ts,
        COUNT(*) OVER (PARTITION BY order_id)          AS qtd_avaliacoes_pedido,
        (COUNT(*) OVER (PARTITION BY order_id) > 1)    AS flag_pedido_multiplas_avaliacoes,
        (COUNT(*) OVER (PARTITION BY review_id) > 1)   AS flag_review_repetida,
        _arquivo_origem,
        _data_ingestao
      FROM workspace.bronze.order_reviews
    """,
    "chave": ["review_id", "order_id"],
    "obrigatorias": ["review_id", "order_id", "review_score", "review_creation_ts"],
    "regras": [
      ("score_fora_de_1_a_5",       "review_score NOT BETWEEN 1 AND 5"),
      ("resposta_antes_da_criacao", "review_answer_ts < review_creation_ts"),
    ],
    "pares_vazios": [
      ("review_score",            "review_score"),
      ("review_comment_title",    "review_comment_title"),
      ("review_comment_message",  "review_comment_message"),
      ("review_creation_date",    "review_creation_ts"),
      ("review_answer_timestamp", "review_answer_ts"),
    ],
    "informativos": [
      ("linhas_pedido_multiplas_avaliacoes", "flag_pedido_multiplas_avaliacoes"),
      ("linhas_review_repetida",             "flag_review_repetida"),
    ],
  },
    "geolocation": {
    "sql": """
      CREATE OR REPLACE TABLE workspace.silver.geolocation AS
      SELECT
        geolocation_zip_code_prefix,
        try_cast(geolocation_lat AS DOUBLE) AS geolocation_lat,
        try_cast(geolocation_lng AS DOUBLE) AS geolocation_lng,
        geolocation_city,
        geolocation_state,
        (try_cast(geolocation_lat AS DOUBLE) < -34
         OR try_cast(geolocation_lat AS DOUBLE) > 6
         OR try_cast(geolocation_lng AS DOUBLE) < -74
         OR try_cast(geolocation_lng AS DOUBLE) > -34) AS flag_fora_do_brasil,
        _arquivo_origem,
        _data_ingestao
      FROM workspace.bronze.geolocation
    """,
    "chave": None,
    "obrigatorias": ["geolocation_zip_code_prefix", "geolocation_lat",
                     "geolocation_lng", "geolocation_state"],
    "regras": [
      ("cep_com_5_digitos",   "length(geolocation_zip_code_prefix) <> 5"),
      ("estado_2_maiusculas", "NOT (geolocation_state RLIKE '^[A-Z]{2}$')"),
    ],
    "pares_vazios": [
      ("geolocation_lat", "geolocation_lat"),
      ("geolocation_lng", "geolocation_lng"),
    ],
    "informativos": [("fora_do_brasil", "flag_fora_do_brasil")],
  },
   "category_translation": {
    "sql": """
      CREATE OR REPLACE TABLE workspace.silver.category_translation AS
      SELECT
        product_category_name,
        product_category_name_english,
        _arquivo_origem,
        _data_ingestao
      FROM workspace.bronze.category_translation
    """,
    "chave": "product_category_name",
    "obrigatorias": ["product_category_name", "product_category_name_english"],
    "regras": [
      ("nome_fora_do_padrao",
       "product_category_name <> lower(trim(product_category_name))"),
    ],
    "pares_vazios": [],
    "informativos": [],
  },
}

In [0]:
def _contar_vazios(tabela, coluna):
    return spark.sql(f"SELECT COUNT(*) - COUNT({coluna}) AS n FROM {tabela}").first()["n"]

def validate_silver(nome, spec):
    b, s = f"workspace.bronze.{nome}", f"workspace.silver.{nome}"
    df = spark.table(s)
    total_b, total_s = spark.table(b).count(), df.count()
    falhas = []
    if total_b != total_s:
        falhas.append(f"contagem: bronze={total_b} silver={total_s}")
    if spec["chave"] is not None:
        cols = spec["chave"] if isinstance(spec["chave"], list) else [spec["chave"]]
        dup = total_s - df.select(*cols).distinct().count()
        if dup != 0:
            falhas.append(f"chave {cols} duplicada: {dup}")
    for col in spec["obrigatorias"]:
        n = df.filter(f"{col} IS NULL").count()
        if n != 0:
            falhas.append(f"{col} vazia: {n}")
    for rotulo, cond in spec["regras"]:
        n = df.filter(cond).count()
        if n != 0:
            falhas.append(f"regra {rotulo}: {n} violações")
    for col_b, col_s in spec["pares_vazios"]:
        vb, vs = _contar_vazios(b, col_b), _contar_vazios(s, col_s)
        if vb != vs:
            falhas.append(f"vazios {col_b}: bronze={vb} silver={vs}")
    info = {rotulo: df.filter(cond).count() for rotulo, cond in spec["informativos"]}
    return total_b, total_s, falhas, info

In [0]:
def process_silver(run_id=None):
    run_id = run_id or str(uuid.uuid4())
    resultados, auditoria = [], []
    for nome, spec in SILVER_TABELAS.items():
        inicio = datetime.now()
        info = {}
        try:
            spark.sql(spec["sql"])
            esperado, linhas, falhas, info = validate_silver(nome, spec)
            status = "OK" if not falhas else "DIVERGENTE"
            erro = None if not falhas else "; ".join(falhas)[:200]
        except Exception as e:
            esperado, linhas, status, erro = None, None, "ERRO", str(e)[:200]
        fim = datetime.now()
        resultados.append((nome, esperado, linhas, status, erro, info))
        auditoria.append((run_id, "silver", nome, inicio, fim,
                          (fim - inicio).total_seconds(),
                          status, esperado, linhas, erro))
    record_audit(auditoria)
    return resultados